# Standard Logging in MPI Codes

Some application codes that use {{poptus}} logging infrastructure use distributed parallelism {{via}} MPI.  This allows users to run the program using two or more distinct MPI processes in order to achieve increased performance.  Parallelization, however, complicates the act of logging information since more than one process can attempt to write to `stdout` or to a file.  In addition, MPI-based codes are often computationally intense and large.  Therefore, the amount of information that could be logged by all processes could be overwhelming if care is not taken.

MPI-based applications that use the {{poptus}} logging facility are presently limited to logging only to `stdout` and `stderr`.  Logging in such applications should be an extension of standard logging in serial applications.  This notebook presents output from an MPI-based application to explain how the output differs from that of standard logging in a serial application.

While the lines written to `stdout` are presented in chronological order, the order in which messages logged by different MPI processes appear could be nondeterministic.

In [1]:
# Remove input/outputs in rendering

from my_mpi_method_hack import my_mpi_method_hack as my_mpi_method

## Default logging level

In many MPI-based applications, logging of general program flow is performed only by a single process.  In the case of {{poptus}}, this process is the MPI process that the application considers to be the lead logging process.  However, all warnings or errors detected by any MPI process are written to, respectively, `stdout` or `stderr`.  This can be important since it allows for a failing process to communicate information before terminating its execution.

To maximize the impact of this example, we consider an application that requires that calling code provide a valid logger rather than simply pass in a valid logger configuration.  In the latter case, the application would create the MPI-aware loggers correctly using the configuration, and would also likely determine which process is the lead logger process.

Since this example logs to `stdout` and `stderr`, the base configuration of the logger is identical to that used in the serial standard logging examples --- users need only specify the log level.  To communicate to {{poptus}} that logging will occur in an MPI environment, each MPI process creates its dedicated logger by passing its rank and indicating if it should be considered as the lead logging process.  While the lead logger's behavior is identical to that of standard logging in serial applications, the loggers provided to worker processes are aware of their secondary logging role.  In particular, worker processes do not log general information ({{ie}} any messages logged at the default log level).

The following logging output is from a simple MPI-based script with the log level set to the default level, which means that no debug information is logged.  The lead process logs using that application's name, \"MpiTest\".  The only logging from a worker process, whose log label is \"Rank 2\", is a warning.  Note that due to the small amount of logging by worker processes and the way that the logging is performed by the method, this log output is deterministic.

In [2]:
import poptus

from mpi4py import MPI

# Hardcoded values
LEAD_PROCESS = 0

# Standard MPI setup
mpi_comm = MPI.COMM_WORLD
rank = mpi_comm.Get_rank()
is_lead = (rank == LEAD_PROCESS)

# Each process creates its MPI-aware logger
configuration = {"Level": poptus.LOG_LEVEL_DEFAULT}
logger = poptus.create_logger(configuration, rank=rank, is_lead=is_lead)

# Call MPI-based function
my_mpi_method(logger, mpi_comm, LEAD_PROCESS)

[MpiTest] Running MPI example with 4 processes
[MpiTest] The lead logging process is rank 0
[MpiTest] Logging at verbosity level 1
[MpiTest] Starting computations
[Rank 2] WARNING - Are you sure that's the task I should work on?
[MpiTest] Computations finished successfully


In [3]:
# Remove input/outputs in rendering

# We are using mpi4py within a single Jupyter book process, with no use of mpirun.
# Check that its values are sufficient for getting this notebook to produce correct
# output.
assert rank == 0
assert is_lead
assert isinstance(logger, poptus.MpiLeadLogger)
assert logger.is_lead_logger

## Minimal debug logging

While for many applications there is no need for MPI worker processes to communicate to users directly during production runs, allowing for workers to directly log debug information can be useful for development and testing.

For our method, we see that debug output helps confirm correct assignment of tasks to the different worker processes.  This increased logging, however, makes the actual ordering of some log lines nondeterministic.

In [4]:
configuration["Level"] = poptus.LOG_LEVEL_MIN_DEBUG
logger = poptus.create_logger(configuration, rank=rank, is_lead=is_lead)

my_mpi_method(logger, mpi_comm, LEAD_PROCESS)

[MpiTest] Running MPI example with 4 processes
[MpiTest] The lead logging process is rank 0
[MpiTest] Logging at verbosity level 2
[MpiTest] Starting computations
[MpiTest] Sending 0.0 task to the rank 1 worker process
[MpiTest] Sending 1.1 task to the rank 2 worker process
[MpiTest] Sending 2.2 task to the rank 3 worker process
[Rank 1] Received task 0.0 from lead process
[Rank 2] Received task 1.1 from lead process
[Rank 2] WARNING - Are you sure that's the task I should work on?
[Rank 3] Received task 2.2 from lead process
[MpiTest] Computations finished successfully
